# BirdCLEF 2026 — v3 Learnable Temporal Smooth

## Concept

Replaces fixed Gaussian kernel with a **learnable per-class 1D convolution** trained on OOF logits.
Also implements **soundscape co-occurrence boost** from BirdCLEF 2024 3rd place (+0.02 LB).

## Experiments

| ID | Method | Key Idea | Init |
|----|--------|----------|------|
| **A** | Learnable Per-Class Conv | `nn.Conv1d(234, 234, K=5, groups=234)` trained on OOF BCE | Gaussian [.1,.2,.4,.2,.1] |
| **B** | Asymmetric Causal Kernel | Aves classes get onset-biased init [.05,.1,.5,.25,.1]; Texture get symmetric | Class-type specific |
| **C** | Co-occurrence Boost | `P += α*(P_max + P_mean - P_max_mean)` per soundscape | α=0.8 (ref: BirdCLEF2024 3rd) |

## Pipeline

```
raw_scores → fuse_scores (prior + texture_smooth + event_smooth)
          → LGBM probe
          → LearnableTemporalSmooth (trained on OOF, per-class Conv1d)
          → sigmoid(/ TEMP_SCALE)
          → co-occurrence boost (Exp C)
          → heads blend → VLOM → tricks
```


In [ ]:
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import gc
import json
import re
import time
import threading
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import timm

from lightgbm import LGBMClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

START = time.time()
print(f"TensorFlow: {tf.__version__}")
print(f"PyTorch: {torch.__version__}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# UPDATE DATASET_DIR to match your Kaggle dataset upload path for submissions_v3
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE             = Path('/kaggle/input/competitions/birdclef-2026')
MODEL_DIR        = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')
DATASET_DIR      = '/kaggle/input/datasets/tom99763/birdclef2026-claude/submissions-v3/submissions_v3/weights'
PERCH_TFLITE     = f'{DATASET_DIR}/perch_v2_cpu.tflite'
HEAD_PSEUDO      = f'{DATASET_DIR}/label_head_pseudo.tflite'
HEAD_SOUNDSCAPE  = f'{DATASET_DIR}/label_head_soundscape_train.tflite'
HEAD_EMBEDDING   = f'{DATASET_DIR}/embedding_head_nohuman_embedding_soundscape.tflite'
PERCH_CACHE_INPUT_DIR = Path('/kaggle/input/datasets/jaejohn/perch-meta')   # attach this dataset
PERCH_CACHE_WORK_DIR  = Path('/kaggle/working/perch_cache')
SILERO_DIR       = f'/kaggle/input/datasets/tom99763/birdclef2026-claude/submission/submission/silero_vad'

SED_CHECKPOINTS = [
    {'name': 'v30-multipseu', 'path': f'{DATASET_DIR}/soup_sed-b0-v30-multipseu.pt',
     'backbone': 'tf_efficientnet_b0.ns_jft_in1k', 'n_mels': 224, 'weight': 1.0},
    {'name': 'v9-asl-soup',   'path': f'{DATASET_DIR}/soup_sed-b0-v9-asl.pt',
     'backbone': 'tf_efficientnet_b0.ns_jft_in1k', 'n_mels': 224, 'weight': 1.0},
    {'name': 'competitor',    'path': f'{DATASET_DIR}/competitor_sed_fold0.pt',
     'backbone': 'tf_efficientnet_b0.ns_jft_in1k', 'n_mels': 224, 'weight': 1.0},
]

# ── Frozen probe params (tuned OOF, do not change) ────────────────────────────
FROZEN_PROBE  = {'pca_dim': 64, 'min_pos': 8, 'C': 0.50, 'alpha': 0.40}
FROZEN_FUSION = {'lambda_event': 0.4, 'lambda_texture': 1.0,
                 'lambda_proxy_texture': 0.8, 'smooth_texture': 0.35}

# ── LGBM probe hyperparameters ────────────────────────────────────────────────
LGBM_PARAMS = {
    'n_estimators': 100,
    'max_depth': 3,
    'num_leaves': 7,
    'learning_rate': 0.05,
    'min_child_samples': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'verbose': -1,
}

# ── Finetuned head weights ────────────────────────────────────────────────────
W_HEAD_PSEUDO     = 1.0    # label_head_pseudo.tflite
W_HEAD_SOUNDSCAPE = 1.0    # label_head_soundscape_train.tflite
W_HEAD_EMBEDDING  = 1.0    # embedding_head_nohuman.tflite
HEAD_BLEND_ALPHA  = 0.30   # 30% finetuned heads, 70% Bayesian+probe

# ── Blend weights ─────────────────────────────────────────────────────────────
PERCH_W         = 0.50
SED_W           = 0.50
USE_VLOM_BLEND  = True

# ── Tricks ────────────────────────────────────────────────────────────────────
TRICK1_FILE_MAX     = True     # +5% file-max into every clip
FILE_MAX_WEIGHT     = 0.05
TRICK2_SMOOTH       = True     # confidence-sharpened temporal smoothing (global)
SHARPEN_POWER       = 1.5
SMOOTH_W5           = np.array([0.05, 0.15, 0.60, 0.15, 0.05])   # 5-tap kernel
SMOOTH_W3           = np.array([0.20, 0.60, 0.20])                # 3-tap fallback
TRICK3_NO_PEAK_NORM = True     # no peak normalization before mel (SED models)
# ── Reference-notebook techniques (0-910, 0-908) ─────────────────────────────
TEMP_SCALE          = 1.15     # temperature scaling on Perch logits before sigmoid (ref: 0-908)
GAUSSIAN_SMOOTH     = True     # Gaussian smooth on Perch LOGITS before sigmoid (ref: 0-910) — enabled
GAUSSIAN_KERNEL     = np.array([0.1, 0.2, 0.4, 0.2, 0.1])        # 0-910 kernel
SMOOTH_EVENT_ALPHA  = 0.15     # event (Aves) local-max smoothing alpha (ref: 0-908)
GAUSSIAN_ON_LOGITS  = True     # True=smooth logits (0-910 style), False=smooth probs (old style)

# ── Constants ─────────────────────────────────────────────────────────────────
NUM_CLASSES    = 234
N_PERCH        = 14795
N_EMBED        = 1536
SR             = 32_000
WINDOW_SAMPLES = SR * 5
CLIP_SAMPLES   = WINDOW_SAMPLES
FILE_SAMPLES   = SR * 60
N_WINDOWS      = 12

import glob
sample_sub     = pd.read_csv(BASE / 'sample_submission.csv')
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
primary_labels = PRIMARY_LABELS   # alias

TEST_DIR  = str(BASE / 'test_soundscapes')
SC_DIR    = TEST_DIR
ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
if not ogg_files:
    ogg_files = sorted(glob.glob(str(BASE / 'train_soundscapes' / '*.ogg')))[:20]
    print(f'Hidden test not found — dry-run on {len(ogg_files)} train soundscapes')

PERCH_CACHE_WORK_DIR.mkdir(parents=True, exist_ok=True)
SED_CHECKPOINTS = [c for c in SED_CHECKPOINTS if os.path.isfile(c['path'])]

print(f'Soundscapes      : {len(ogg_files)}')
print(f'SED models       : {[c["name"] for c in SED_CHECKPOINTS]}')
print(f'T1_file_max={TRICK1_FILE_MAX}  T2_smooth={TRICK2_SMOOTH}  T3_no_peak_norm={TRICK3_NO_PEAK_NORM}')
print(f'Blend: Perch={PERCH_W}  SED={SED_W}  VLOM={USE_VLOM_BLEND}')
# ── Learnable Temporal Smooth (NEW) ─────────────────────────────────────────
LEARN_SMOOTH_KERNEL_SIZE  = 5       # filter width (frames); must be odd
LEARN_SMOOTH_LR           = 0.05    # learning rate for kernel training
LEARN_SMOOTH_EPOCHS       = 30      # training epochs on OOF logits
LEARN_SMOOTH_L2           = 1e-3    # L2 regularization on kernel weights
LEARN_SMOOTH_INIT         = 'gaussian'  # 'gaussian' | 'asymmetric' | 'identity'
# Co-occurrence boost (BirdCLEF 2024 3rd place, +0.02 LB)
COOCCUR_BOOST_ALPHA       = 0.8     # weight for file-level max/mean boost
COOCCUR_BOOST_ENABLED     = True

print(f'TempScale={TEMP_SCALE}  GaussianSmooth={GAUSSIAN_SMOOTH}')
print(f'LearnSmooth: kernel={LEARN_SMOOTH_KERNEL_SIZE}  epochs={LEARN_SMOOTH_EPOCHS}  init={LEARN_SMOOTH_INIT}')
print(f'CooccurBoost: enabled={COOCCUR_BOOST_ENABLED}  alpha={COOCCUR_BOOST_ALPHA}')

In [ ]:
# ── SED Model Definitions ──────────

@dataclass
class SEDConfig:
    sr: int = 32_000; n_mels: int = 224; n_fft: int = 2048; hop_length: int = 512
    fmin: int = 0; fmax: int = 16_000; top_db: float = 80.0; power: float = 2.0
    norm: str = 'slaney'; mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234; in_channels: int = 3
    dropout: float = 0.1; drop_path_rate: float = 0.0; gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init)); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = self.fc(x.permute(0,2,1)).permute(0,2,1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        return {'clipwise_prob': torch.sigmoid((att * cls).sum(-1)),
                'clipwise_logit': (att * cls).sum(-1)}


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
                                          features_only=False, global_pool='', num_classes=0,
                                          drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))


class MelTransform(nn.Module):
    """Mel transform — TRICK 3: skip peak normalization if peak_norm=False."""
    def __init__(self, cfg, peak_norm=True):
        super().__init__()
        self.peak_norm = peak_norm
        self.mel = T.MelSpectrogram(sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
                                    n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
                                    power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        if self.peak_norm:
            peak = waveforms.abs().amax(dim=1, keepdim=True).clamp(min=1e-7)
            waveforms = waveforms / peak
        mel = torch.nan_to_num(self.db(self.mel(waveforms)), nan=-80.0)
        B = mel.shape[0]; flat = mel.reshape(B, -1)
        mn = flat.min(1, keepdim=True)[0].unsqueeze(-1)
        mx = flat.max(1, keepdim=True)[0].unsqueeze(-1)
        mel = torch.nan_to_num((mel - mn) / (mx - mn + 1e-7), nan=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)

print('SED classes defined.')

In [ ]:
# ── Perch taxonomy + mapping + proxy setup ──

taxonomy          = pd.read_csv(BASE / 'taxonomy.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

taxonomy['primary_label']          = taxonomy['primary_label'].astype(str)
soundscape_labels['primary_label'] = soundscape_labels['primary_label'].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(';') if t.strip()]

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'file_id': None, 'site': None, 'date': pd.NaT,
                'time_utc': None, 'hour_utc': -1, 'month': -1}
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce')
    return {'file_id': file_id, 'site': site, 'date': dt,
            'time_utc': hms, 'hour_utc': int(hms[:2]),
            'month': int(dt.month) if pd.notna(dt) else -1}

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)

sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec']   = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id']    = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)

meta_sc = sc_clean['filename'].apply(lambda fn: parse_soundscape_filename(fn)).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta_sc], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
Y_SC = np.zeros((len(sc_clean), NUM_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean['label_list']):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth['index'].to_numpy()]

# ── TFLite Perch: thread-local interpreter ────────────────────────────────────
_tls = threading.local()

def get_perch_tflite():
    if not hasattr(_tls, 'perch'):
        p = tf.lite.Interpreter(model_path=PERCH_TFLITE, num_threads=1)
        p.allocate_tensors()
        p_out = p.get_output_details()
        p_lbl_idx = next(i for i, o in enumerate(p_out) if o['shape'][-1] == N_PERCH)
        p_emb_idx = next(i for i, o in enumerate(p_out) if o['shape'][-1] == N_EMBED)
        _tls.perch = p
        _tls.p_inp = p.get_input_details()[0]['index']
        _tls.p_lbl = p_out[p_lbl_idx]['index']
        _tls.p_emb = p_out[p_emb_idx]['index']
        import os as _os
        for _attr, _path in [('h1', HEAD_PSEUDO), ('h2', HEAD_SOUNDSCAPE)]:
            if _os.path.isfile(_path):
                _h = tf.lite.Interpreter(model_path=_path, num_threads=1)
                _h.allocate_tensors()
                setattr(_tls, _attr, _h)
                setattr(_tls, _attr+'_inp', _h.get_input_details()[0]['index'])
                setattr(_tls, _attr+'_out', _h.get_output_details()[0]['index'])
        if _os.path.isfile(HEAD_EMBEDDING):
            _h3 = tf.lite.Interpreter(model_path=HEAD_EMBEDDING, num_threads=1)
            _h3.allocate_tensors()
            _tls.h3     = _h3
            _tls.h3_inp = _h3.get_input_details()[0]['index']
            _tls.h3_out = _h3.get_output_details()[0]['index']
    return _tls

# ── Mapping ───────────────────────────────────────────────────────────────────
bc_labels_df = (
    pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
    .reset_index()
    .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
)

MANUAL_SCIENTIFIC_NAME_MAP = {}
taxonomy_copy = taxonomy.copy()
taxonomy_copy['scientific_name_lookup'] = taxonomy_copy['scientific_name'].replace(MANUAL_SCIENTIFIC_NAME_MAP)

bc_lookup = bc_labels_df.rename(columns={'scientific_name': 'scientific_name_lookup'})
mapping = taxonomy_copy.merge(
    bc_lookup[['scientific_name_lookup', 'bc_index']],
    on='scientific_name_lookup', how='left'
)

NO_LABEL_INDEX = len(bc_labels_df)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index('primary_label')['bc_index']
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK       = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS        = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS      = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()
TEXTURE_TAXA   = {'Amphibia', 'Insecta'}

ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32)

idx_mapped_active_texture  = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event    = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive       = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32)

unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)
].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits  = bc_labels_df[
        bc_labels_df['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row['primary_label']
    sci    = row['scientific_name']
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            'target_scientific_name': sci, 'genus': genus,
            'bc_indices': hits['bc_index'].astype(int).tolist(),
            'proxy_scientific_names': hits['scientific_name'].tolist(),
        }

SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys() if CLASS_NAME_MAP.get(t) == 'Amphibia'
])
selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]['bc_indices'], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture    = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event   = np.setdiff1d(idx_unmapped_active_event,   selected_proxy_pos)

print(f'Mapped classes: {MAPPED_MASK.sum()} / {NUM_CLASSES}')
print(f'Unmapped classes: {(~MAPPED_MASK).sum()}')
print('TFLite Perch thread-local helper: get_perch_tflite()')

In [ ]:
# ── Helper functions ──────────────────

def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')


def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x      = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def smooth_events_fixed12(scores, cols, alpha=0.15):
    """Local-max propagation for Aves (event) classes. Ref: 0-908 (0.908 LB).
    Birds produce transient calls — averaging neighbours weakens peak signal.
    Local-max propagates detections to adjacent windows without diluting them.
    """
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x    = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev, nxt))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s


def seq_features_1d(v):
    assert len(v) % N_WINDOWS == 0
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v  = np.repeat(x.max(axis=1),  N_WINDOWS)
    return prev_v, next_v, mean_v, max_v


def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    site_p    = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = prior_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p    = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = prior_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_prior[mask].mean(axis=0)

    sh_to_i = {}; sh_n_list = []; sh_p_list = []
    for (s, h), idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx)); sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = (np.stack(sh_p_list).astype(np.float32)
            if len(sh_p_list) else np.zeros((0, Y_prior.shape[1]), dtype=np.float32))

    return {'global_p': global_p, 'site_to_i': site_to_i, 'site_n': site_n, 'site_p': site_p,
            'hour_to_i': hour_to_i, 'hour_n': hour_n, 'hour_p': hour_p,
            'sh_to_i': sh_to_i, 'sh_n': sh_n, 'sh_p': sh_p}


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables['global_p'][None, :], n, axis=0).astype(np.float32, copy=True)
    site_idx = np.fromiter((tables['site_to_i'].get(str(s), -1) for s in sites), dtype=np.int32, count=n)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(h), -1) if int(h) >= 0 else -1 for h in hours), dtype=np.int32, count=n)
    sh_idx   = np.fromiter((tables['sh_to_i'].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)), dtype=np.int32, count=n)
    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]; wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]
    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]; ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]
    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]; wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]
    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def fuse_scores_with_tables(
    base_scores, sites, hours, tables,
    lambda_event=FROZEN_FUSION['lambda_event'],
    lambda_texture=FROZEN_FUSION['lambda_texture'],
    lambda_proxy_texture=FROZEN_FUSION['lambda_proxy_texture'],
    smooth_texture=FROZEN_FUSION['smooth_texture'],
    smooth_event=SMOOTH_EVENT_ALPHA,
):
    scores = base_scores.copy()
    prior  = prior_logits_from_tables(sites, hours, tables)
    if len(idx_mapped_active_event):    scores[:, idx_mapped_active_event]    += lambda_event   * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):  scores[:, idx_mapped_active_texture]  += lambda_texture * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture): scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):   scores[:, idx_selected_prioronly_active_event]   = lambda_event   * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture): scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):      scores[:, idx_unmapped_inactive]      = -8.0
    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)  # 0-908: local-max for Aves
    return scores.astype(np.float32, copy=False), prior


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """
    LGBM version: 74-dim features (vs 71-dim in base v3-ensemble).
    emb_proj: (n, 64)  → PCA embedding
    raw_col, prior_col, base_col: (n,)
    Added: 3 interaction features (raw×prior, raw×base, prior×base)
    returns: (n, 74)
    """
    prev_base, next_base, mean_base, max_base = seq_features_1d(base_col)
    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        # Interaction features (from perch-v2-starter-train-infer-mlp-0-905)
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)
    return feats.astype(np.float32, copy=False)


print('Helper functions defined. build_class_features → 74-dim (LGBM).')

In [ ]:
# ── Perch inference + cache loading ──────────────────────────────────────────

def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr != SR: raise ValueError(f'Unexpected sample rate {sr}')
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES: y = y[:FILE_SAMPLES]
    return y


def _run_tflite_clips(x_clips, proxy_reduce='max'):
    n = len(x_clips)
    logits = np.zeros((n, N_PERCH), dtype=np.float32)
    emb    = np.zeros((n, N_EMBED), dtype=np.float32)
    tls = get_perch_tflite()
    for ci in range(n):
        tls.perch.set_tensor(tls.p_inp, x_clips[ci:ci + 1])
        tls.perch.invoke()
        logits[ci] = tls.perch.get_tensor(tls.p_lbl)[0]
        emb[ci]    = tls.perch.get_tensor(tls.p_emb)[0]
    return logits, emb


def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce='max'):
    paths  = [Path(p) for p in paths]
    n_files = len(paths); n_rows = n_files * N_WINDOWS
    row_ids = np.empty(n_rows, dtype=object); filenames = np.empty(n_rows, dtype=object)
    sites   = np.empty(n_rows, dtype=object); hours     = np.empty(n_rows, dtype=np.int16)
    scores  = np.zeros((n_rows, NUM_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, N_EMBED), dtype=np.float32)
    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose: iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc='Perch batches')
    for start in iterator:
        batch_paths = paths[start:start + batch_files]; batch_n = len(batch_paths)
        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row; x_pos = 0
        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
            meta_p = parse_soundscape_filename(path.name); stem = path.stem
            row_ids[write_row:write_row + N_WINDOWS]   = [f'{stem}_{t}' for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS]     = meta_p['site']
            hours[write_row:write_row + N_WINDOWS]     = int(meta_p['hour_utc'])
            x_pos += N_WINDOWS; write_row += N_WINDOWS
        logits, emb = _run_tflite_clips(x, proxy_reduce=proxy_reduce)
        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            scores[batch_row_start:write_row, pos] = (sub.max(axis=1) if proxy_reduce == 'max' else sub.mean(axis=1)).astype(np.float32)
        del x, logits, emb; gc.collect()
    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours})
    return meta_df, scores, embeddings


def build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=True):
    groups_full = meta_full['filename'].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)
    oof_base  = np.zeros_like(scores_full_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)
    fold_id   = np.full(len(meta_full), -1, dtype=np.int16)
    for fold, (tr_idx, va_idx) in enumerate(tqdm(list(gkf.split(scores_full_raw, groups=groups_full)), desc='OOF', disable=not verbose), 1):
        tr_idx = np.sort(tr_idx); va_idx = np.sort(va_idx)
        val_files = set(meta_full.iloc[va_idx]['filename'].tolist())
        prior_mask = ~sc_clean['filename'].isin(val_files).values
        tables = fit_prior_tables(sc_clean.loc[prior_mask].reset_index(drop=True), Y_SC[prior_mask])
        va_base, va_prior = fuse_scores_with_tables(scores_full_raw[va_idx], sites=meta_full.iloc[va_idx]['site'].to_numpy(), hours=meta_full.iloc[va_idx]['hour_utc'].to_numpy(), tables=tables)
        oof_base[va_idx] = va_base; oof_prior[va_idx] = va_prior; fold_id[va_idx] = fold
    assert (fold_id >= 0).all()
    return oof_base, oof_prior, fold_id


# ── Load or build Perch cache ─────────────────────────────────────────────────
def resolve_full_cache_paths():
    candidates = [
        (PERCH_CACHE_WORK_DIR / 'full_perch_meta.parquet', PERCH_CACHE_WORK_DIR / 'full_perch_arrays.npz'),
        (Path('/kaggle/working/full_perch_meta.parquet'), Path('/kaggle/working/full_perch_arrays.npz')),
    ]
    if PERCH_CACHE_INPUT_DIR.exists():
        candidates.append((
            PERCH_CACHE_INPUT_DIR / 'full_perch_meta.parquet',
            PERCH_CACHE_INPUT_DIR / 'full_perch_arrays.npz',
        ))
    for meta_path, npz_path in candidates:
        if meta_path.exists() and npz_path.exists():
            return meta_path, npz_path
    return None, None


cache_meta, cache_npz = resolve_full_cache_paths()

if cache_meta is not None and cache_npz is not None:
    print('Loading cached full-file Perch outputs from:', cache_meta)
    meta_full       = pd.read_parquet(cache_meta)
    arr             = np.load(cache_npz)
    scores_full_raw = arr['scores_full_raw'].astype(np.float32)
    emb_full        = arr['emb_full'].astype(np.float32)
else:
    print('No cache found. Running Perch on trusted full files...')
    full_paths = [BASE / 'train_soundscapes' / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_with_embeddings(full_paths, batch_files=16, verbose=True)
    out_meta = PERCH_CACHE_WORK_DIR / 'full_perch_meta.parquet'
    out_npz  = PERCH_CACHE_WORK_DIR / 'full_perch_arrays.npz'
    meta_full.to_parquet(out_meta, index=False)
    np.savez_compressed(out_npz, scores_full_raw=scores_full_raw, emb_full=emb_full)
    print('Saved cache to:', out_npz)

full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()]

print('meta_full:', meta_full.shape)
print('scores_full_raw:', scores_full_raw.shape)
print('emb_full:', emb_full.shape)
print('Y_FULL:', Y_FULL.shape)

# ── Build OOF meta-features ───────────────────────────────────────────────────
OOF_META_CACHE = PERCH_CACHE_WORK_DIR / 'full_oof_meta_features.npz'

if OOF_META_CACHE.exists():
    print('Loading cached OOF meta-features from:', OOF_META_CACHE)
    arr_oof    = np.load(OOF_META_CACHE)
    oof_base   = arr_oof['oof_base'].astype(np.float32)
    oof_prior  = arr_oof['oof_prior'].astype(np.float32)
    oof_fold_id = arr_oof['fold_id'].astype(np.int16)
elif (PERCH_CACHE_INPUT_DIR / 'full_oof_meta_features.npz').exists():
    print('Loading OOF meta-features from PERCH_CACHE_INPUT_DIR')
    arr_oof    = np.load(PERCH_CACHE_INPUT_DIR / 'full_oof_meta_features.npz')
    oof_base   = arr_oof['oof_base'].astype(np.float32)
    oof_prior  = arr_oof['oof_prior'].astype(np.float32)
    oof_fold_id = arr_oof['fold_id'].astype(np.int16)
else:
    print('Building OOF meta-features...')
    oof_base, oof_prior, oof_fold_id = build_oof_base_prior(
        scores_full_raw=scores_full_raw, meta_full=meta_full,
        sc_clean=sc_clean, Y_SC=Y_SC, n_splits=5, verbose=True
    )
    np.savez_compressed(OOF_META_CACHE, oof_base=oof_base, oof_prior=oof_prior, fold_id=oof_fold_id)
    print('Saved OOF meta-features to:', OOF_META_CACHE)

# ── Fit final prior tables ────────────────────────────────────────────────────
final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)
print('Built final prior tables for inference.')

# ── Fit embedding scaler + PCA ────────────────────────────────────────────────
emb_scaler      = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)
n_comp_final    = min(int(FROZEN_PROBE['pca_dim']), emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca         = PCA(n_components=n_comp_final)
Z_FULL          = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)
print('Z_FULL:', Z_FULL.shape, '  explained variance:', emb_pca.explained_variance_ratio_.sum())

# ── Train final classwise LGBM probe models ───────────────────────────────────
full_pos_counts = Y_FULL.sum(axis=0)
PROBE_CLASS_IDX = np.where(full_pos_counts >= int(FROZEN_PROBE['min_pos']))[0].astype(np.int32)
probe_models    = {}

for cls_idx in tqdm(PROBE_CLASS_IDX, desc='Training LGBM probes'):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue

    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )

    n_pos = y.sum()
    n_neg = len(y) - n_pos
    spw   = float(n_neg) / max(float(n_pos), 1.0)

    clf = LGBMClassifier(
        **LGBM_PARAMS,
        scale_pos_weight=spw,
    )
    clf.fit(X_cls, y)
    probe_models[cls_idx] = clf

print(f'Trained final LGBM probe models: {len(probe_models)}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Learnable Per-Class Temporal Smooth
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class LearnableTemporalSmooth(nn.Module):
    """
    Learnable per-class 1D convolution over temporal logit sequences.
    
    Architecture:
      - nn.Conv1d(C, C, kernel_size=K, groups=C, padding=K//2, bias=False)
      - Each of the 234 classes has its own independent K-tap FIR filter
      - Total parameters: 234 × K (1170 for K=5)
    
    Initialization options:
      'gaussian'   : [0.1, 0.2, 0.4, 0.2, 0.1] — warm start from 0-910 kernel
      'asymmetric' : Aves=[0.05,0.10,0.50,0.25,0.10], Texture=[0.10,0.20,0.40,0.20,0.10]
                     Aves init biases toward onset (left-heavy forward-looking)
      'identity'   : center=1, others=0 — conservative start
    """
    def __init__(self, n_classes, kernel_size, init_mode='gaussian',
                 event_indices=None, texture_indices=None):
        super().__init__()
        self.n_classes   = n_classes
        self.kernel_size = kernel_size
        self.padding     = kernel_size // 2
        self.conv = nn.Conv1d(n_classes, n_classes, kernel_size,
                              groups=n_classes, padding=self.padding,
                              bias=False, padding_mode='replicate')
        self._init_weights(init_mode, event_indices, texture_indices)

    def _init_weights(self, mode, event_idx, texture_idx):
        K = self.kernel_size
        with torch.no_grad():
            if mode == 'gaussian':
                kernel = torch.tensor([0.1, 0.2, 0.4, 0.2, 0.1], dtype=torch.float32)
                if K != 5:
                    # Fallback: normalized uniform
                    kernel = torch.ones(K) / K
                self.conv.weight.data[:, 0, :] = kernel.unsqueeze(0).expand(self.n_classes, -1)

            elif mode == 'asymmetric':
                # Default: symmetric Gaussian
                sym_k = torch.tensor([0.1, 0.2, 0.4, 0.2, 0.1], dtype=torch.float32)
                # Aves (event): onset-biased — past context heavier than future
                # [t-2, t-1, t, t+1, t+2] weights: more weight on t-1, t, t+1
                # BirdCLEF files are read L→R so "past" = earlier in file
                aves_k = torch.tensor([0.05, 0.10, 0.50, 0.25, 0.10], dtype=torch.float32)
                aves_k = aves_k / aves_k.sum()  # normalize

                # Initialize all with symmetric
                self.conv.weight.data[:, 0, :] = sym_k.unsqueeze(0).expand(self.n_classes, -1)
                # Override Aves classes with asymmetric kernel
                if event_idx is not None and len(event_idx) > 0:
                    ev = torch.tensor(event_idx, dtype=torch.long)
                    self.conv.weight.data[ev, 0, :] = aves_k.unsqueeze(0).expand(len(ev), -1)

            elif mode == 'identity':
                nn.init.zeros_(self.conv.weight)
                self.conv.weight.data[:, 0, K // 2] = 1.0

    def forward(self, logits):
        """
        logits: (N, C) where N = n_files * N_WINDOWS
        Returns smoothed logits of same shape.
        """
        N, C = logits.shape
        n_files = N // N_WINDOWS
        x = torch.tensor(logits, dtype=torch.float32)
        x = x.view(n_files, N_WINDOWS, C).permute(0, 2, 1)  # (n_files, C, T)
        x = self.conv(x)                                      # (n_files, C, T)
        x = x.permute(0, 2, 1).reshape(N, C)
        return x.detach().numpy()


def train_learnable_smooth(oof_base, oof_prior, Y_FULL, scores_full_raw, Z_FULL,
                            kernel_size=5, init_mode='gaussian', lr=0.05,
                            epochs=30, l2=1e-3, event_idx=None, texture_idx=None):
    """
    Train LearnableTemporalSmooth on OOF final_scores using BCE loss.
    
    Returns trained model (eval mode).
    
    Steps:
    1. Compute final logit scores from OOF (same as inference path but OOF)
    2. Apply LGBM probe predictions on OOF to get per-class adjusted logits
    3. Train Conv1d to minimize BCE(sigmoid(smooth(logit)), Y)
    """
    print(f'Training LearnableTemporalSmooth (init={init_mode}, K={kernel_size}, epochs={epochs})...')

    # ── Build OOF final logit scores (replicate inference path) ───────────────
    # Use oof_base as starting point (already has prior fusion + event smooth)
    oof_final = oof_base.copy().astype(np.float32)  # (N, 234)
    alpha = float(FROZEN_PROBE['alpha'])

    # Apply probe predictions to build oof_final (re-use trained probe_models)
    for cls_idx, clf in tqdm(probe_models.items(), desc='OOF probe predictions', leave=False):
        X_cls = build_class_features(
            Z_FULL,
            raw_col=scores_full_raw[:, cls_idx],
            prior_col=oof_prior[:, cls_idx],
            base_col=oof_base[:, cls_idx],
        )
        proba = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
        proba_clipped = np.clip(proba, 1e-7, 1.0 - 1e-7)
        pred = np.log(proba_clipped / (1.0 - proba_clipped))
        oof_final[:, cls_idx] = (1.0 - alpha) * oof_base[:, cls_idx] + alpha * pred

    # ── Setup model and optimizer ─────────────────────────────────────────────
    smooth_model = LearnableTemporalSmooth(
        n_classes=NUM_CLASSES, kernel_size=kernel_size, init_mode=init_mode,
        event_indices=event_idx, texture_indices=texture_idx
    )
    optimizer = torch.optim.Adam(smooth_model.parameters(), lr=lr, weight_decay=l2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    X_logits = torch.tensor(oof_final, dtype=torch.float32)
    Y_target = torch.tensor(Y_FULL, dtype=torch.float32)

    best_loss = float('inf')
    best_state = None

    smooth_model.train()
    for ep in range(epochs):
        optimizer.zero_grad()
        # Forward: smooth logits → sigmoid → BCE
        N, C = X_logits.shape
        n_files = N // N_WINDOWS
        x = X_logits.view(n_files, N_WINDOWS, C).permute(0, 2, 1)
        x = smooth_model.conv(x).permute(0, 2, 1).reshape(N, C)
        # Temperature scaling
        probs = torch.sigmoid(x / TEMP_SCALE)
        # BCE loss (mean over valid classes only)
        loss = nn.functional.binary_cross_entropy(probs, Y_target, reduction='mean')
        loss.backward()
        # Clip gradients for stability
        nn.utils.clip_grad_norm_(smooth_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f'  ep {ep+1:3d}/{epochs}  loss={loss.item():.5f}  lr={scheduler.get_last_lr()[0]:.4f}')
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.clone() for k, v in smooth_model.state_dict().items()}

    # Load best weights
    smooth_model.load_state_dict(best_state)
    smooth_model.eval()

    # ── Evaluate OOF AUC with and without smooth ──────────────────────────────
    with torch.no_grad():
        N, C = X_logits.shape
        n_files = N // N_WINDOWS
        x = X_logits.view(n_files, N_WINDOWS, C).permute(0, 2, 1)
        x_sm = smooth_model.conv(x).permute(0, 2, 1).reshape(N, C)
        probs_smooth = torch.sigmoid(x_sm / TEMP_SCALE).numpy()
        probs_base   = torch.sigmoid(X_logits / TEMP_SCALE).numpy()

    auc_base   = macro_auc_skip_empty(Y_FULL, probs_base)
    auc_smooth = macro_auc_skip_empty(Y_FULL, probs_smooth)
    print(f'OOF AUC — base: {auc_base:.4f}  +smooth: {auc_smooth:.4f}  Δ={auc_smooth-auc_base:+.4f}')

    return smooth_model


# ── Train Experiment A: Gaussian init (warm start from 0-910) ─────────────────
smooth_A = train_learnable_smooth(
    oof_base, oof_prior, Y_FULL, scores_full_raw, Z_FULL,
    kernel_size=LEARN_SMOOTH_KERNEL_SIZE, init_mode='gaussian',
    lr=LEARN_SMOOTH_LR, epochs=LEARN_SMOOTH_EPOCHS, l2=LEARN_SMOOTH_L2,
    event_idx=idx_active_event, texture_idx=idx_active_texture,
)

# ── Train Experiment B: Asymmetric init (Aves onset-biased) ──────────────────
if LEARN_SMOOTH_INIT == 'asymmetric':
    smooth_model_active = train_learnable_smooth(
        oof_base, oof_prior, Y_FULL, scores_full_raw, Z_FULL,
        kernel_size=LEARN_SMOOTH_KERNEL_SIZE, init_mode='asymmetric',
        lr=LEARN_SMOOTH_LR, epochs=LEARN_SMOOTH_EPOCHS, l2=LEARN_SMOOTH_L2,
        event_idx=idx_active_event, texture_idx=idx_active_texture,
    )
else:
    smooth_model_active = smooth_A  # use Gaussian init by default

print('Learnable smooth trained. Active model:', LEARN_SMOOTH_INIT)


In [ ]:
# ── Load SED models ───────────────────────────────────────────────────────────
_sed_entries = []
_SED_LOCK    = threading.Lock()
_peak_norm   = not TRICK3_NO_PEAK_NORM
print(f'TRICK 3: peak_norm={_peak_norm}')

for c in SED_CHECKPOINTS:
    if not os.path.isfile(c['path']):
        print(f'  SKIP {c["name"]}: file not found'); continue
    cfg   = SEDConfig(backbone=c['backbone'], n_mels=c['n_mels'])
    model = SEDModel(cfg)
    ckpt  = torch.load(c['path'], map_location='cpu', weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    if any('freq_pool' in k for k in state):
        state = {k.replace('freq_pool', 'gem_pool'): v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval()
    mel_tf = MelTransform(cfg, peak_norm=_peak_norm)
    mel_tf.eval()
    with torch.no_grad(): model(mel_tf(torch.zeros(1, CLIP_SAMPLES)))
    ep  = ckpt.get('epoch', '?'); auc = ckpt.get('metrics', {}).get('macro_auc', '?')
    print(f'  {c["name"]}  epoch={ep}  val_auc={auc}')
    _sed_entries.append((model, mel_tf, c['weight'], c['name']))

USE_SED = len(_sed_entries) > 0
_W_SED  = sum(w for _, _, w, _ in _sed_entries) if _sed_entries else 1.0
print(f'{len(_sed_entries)} SED models loaded  USE_SED={USE_SED}')


def sed_predict_clips(clips_np):
    if not _sed_entries: return np.zeros((clips_np.shape[0], NUM_CLASSES), dtype=np.float32)
    t   = torch.from_numpy(clips_np).float()
    avg = np.zeros((clips_np.shape[0], NUM_CLASSES), dtype=np.float32)
    with _SED_LOCK, torch.no_grad():
        for model, mel_tf, w, _ in _sed_entries:
            avg += w * model(mel_tf(t))['clipwise_prob'].numpy()
    return avg / _W_SED


In [ ]:
# ── Silero VAD ─────────────────────
_vad_model, _vad_utils = torch.hub.load(
    repo_or_dir=SILERO_DIR, model='silero_vad', source='local', verbose=False
)
get_speech_timestamps = _vad_utils[0]
_VAD_LOCK = threading.Lock()
print('Silero VAD loaded')


def remove_human_voice(audio, sr=32_000):
    t16 = torch.tensor(librosa.resample(audio, orig_sr=sr, target_sr=16_000))
    with _VAD_LOCK:
        segs = get_speech_timestamps(t16, _vad_model, threshold=0.4, sampling_rate=16_000)
    for seg in segs:
        start_s = seg['start'] / 16_000; dur_s = (seg['end'] - seg['start']) / 16_000
        if dur_s >= 2.0 and start_s >= 8.0:
            return audio[:int(start_s * sr)]
    return audio


In [ ]:
# ── Post-processing + VLOM blend functions ────────────────────────────────────

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88, 88)))


def vlom_blend(a, b, w_a=PERCH_W, w_b=SED_W):
    eps = 1e-9; total = w_a + w_b; pa = w_a / total; pb = w_b / total
    geomean = np.exp(pa * np.log(a + eps) + pb * np.log(b + eps))
    rms     = np.sqrt(pa * a ** 2 + pb * b ** 2)
    return ((geomean + rms) / 2.0).astype(np.float32)


def apply_texture_smooth_per_file(scores_12):
    if len(idx_active_texture) == 0: return scores_12
    alpha = FROZEN_FUSION['smooth_texture']; s = scores_12.copy()
    x = s[:, idx_active_texture]
    prev_x = np.concatenate([x[:1, :], x[:-1, :]], axis=0)
    next_x = np.concatenate([x[1:, :], x[-1:, :]], axis=0)
    s[:, idx_active_texture] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def apply_file_max(preds_12):
    if TRICK1_FILE_MAX:
        file_max = np.max(preds_12, axis=0, keepdims=True)
        return preds_12 + FILE_MAX_WEIGHT * file_max
    return preds_12


def apply_trick2(preds):
    n = preds.shape[0]
    if not TRICK2_SMOOTH or n <= 2: return preds
    p_sharp = preds ** SHARPEN_POWER
    if n > 4:
        p_pad    = np.pad(p_sharp, ((2, 2), (0, 0)), mode='edge')
        smoothed = sum(SMOOTH_W5[j] * p_pad[j:j + n] for j in range(5))
    else:
        p_pad    = np.pad(p_sharp, ((1, 1), (0, 0)), mode='edge')
        smoothed = sum(SMOOTH_W3[j] * p_pad[j:j + n] for j in range(3))
    return (smoothed ** (1.0 / SHARPEN_POWER)).astype(np.float32)


def apply_gaussian_smooth(preds):
    """Gaussian temporal smooth [0.1, 0.2, 0.4, 0.2, 0.1] — legacy prob-space version.
    Kept for reference. Prefer gauss_smooth_logits() for logit-space smoothing (0-910)."""
    n = preds.shape[0]
    if n <= 2: return preds
    if n > 4:
        p_pad    = np.pad(preds, ((2, 2), (0, 0)), mode='edge')
        smoothed = sum(GAUSSIAN_KERNEL[j] * p_pad[j:j + n] for j in range(5))
    else:
        k3 = np.array([0.25, 0.5, 0.25])
        p_pad    = np.pad(preds, ((1, 1), (0, 0)), mode='edge')
        smoothed = sum(k3[j] * p_pad[j:j + n] for j in range(3))
    return smoothed.astype(np.float32)


def gauss_smooth_logits(logits):
    """Gaussian temporal smooth on LOGITS before sigmoid. Ref: 0-910 (LB 0.910).
    Uses convolve1d with kernel [0.1, 0.2, 0.4, 0.2, 0.1] per file, mode='nearest'.
    Applied in logit space — stronger effect than prob-space smoothing.
    """
    from scipy.ndimage import convolve1d
    n_files = logits.shape[0] // N_WINDOWS
    smoothed = logits.reshape(n_files, N_WINDOWS, logits.shape[1]).copy()
    for i in range(n_files):
        smoothed[i] = convolve1d(smoothed[i], GAUSSIAN_KERNEL, axis=0, mode='nearest')
    return smoothed.reshape(-1, logits.shape[1]).astype(np.float32)


print('Post-processing + VLOM + TRICK2 + Gaussian smooth functions defined.')

In [ ]:
# ── Main inference loop (LGBM probe: predict_proba → logit → blend) ──────────

def predict_file(ogg_path):
    ogg_path = Path(ogg_path)
    ss_id    = ogg_path.stem
    row_ids  = [f'{ss_id}_{t}' for t in range(5, 65, 5)]
    zeros    = np.zeros((N_WINDOWS, NUM_CLASSES), dtype=np.float32)

    try:
        audio, _ = librosa.load(str(ogg_path), sr=SR, mono=True)
    except Exception as e:
        print(f'  ERROR loading {ogg_path.name}: {e}'); return row_ids, zeros

    clips = np.zeros((N_WINDOWS, CLIP_SAMPLES), dtype=np.float32)
    for i in range(N_WINDOWS):
        start = i * CLIP_SAMPLES; end = start + CLIP_SAMPLES
        chunk = audio[start:end]
        if len(chunk) < CLIP_SAMPLES: chunk = np.pad(chunk, (0, CLIP_SAMPLES - len(chunk)))
        clips[i] = chunk

    try:
        logits_14795, emb_1536 = _run_tflite_clips(clips)

        raw_scores = np.zeros((N_WINDOWS, NUM_CLASSES), dtype=np.float32)
        raw_scores[:, MAPPED_POS] = logits_14795[:, MAPPED_BC_INDICES]
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            raw_scores[:, pos] = logits_14795[:, bc_idx_arr].max(axis=1)

        meta_p   = parse_soundscape_filename(ogg_path.name)
        sites_12 = [meta_p['site']] * N_WINDOWS
        hours_12 = [meta_p['hour_utc']] * N_WINDOWS

        base_scores, prior_scores = fuse_scores_with_tables(
            raw_scores, sites_12, hours_12, final_prior_tables
        )

        emb_scaled = emb_scaler.transform(emb_1536)
        Z_test_12  = emb_pca.transform(emb_scaled).astype(np.float32)

        final_scores = base_scores.copy()
        alpha = float(FROZEN_PROBE['alpha'])
        for cls_idx, clf in probe_models.items():
            X_cls = build_class_features(
                Z_test_12,
                raw_col=raw_scores[:, cls_idx],
                prior_col=prior_scores[:, cls_idx],
                base_col=base_scores[:, cls_idx],
            )
            # LGBM: predict_proba → logit → blend
            proba = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
            proba_clipped = np.clip(proba, 1e-7, 1.0 - 1e-7)
            pred  = np.log(proba_clipped / (1.0 - proba_clipped))  # logit
            final_scores[:, cls_idx] = (
                (1.0 - alpha) * base_scores[:, cls_idx] + alpha * pred
            )

        # Learnable per-class temporal smooth on logits (Exp A/B: trained Conv1d)
        _logits_to_sigmoid = smooth_model_active.forward(final_scores)

        # Fallback Gaussian smooth if learnable model not trained
        if GAUSSIAN_SMOOTH and GAUSSIAN_ON_LOGITS:
            _logits_to_sigmoid = gauss_smooth_logits(_logits_to_sigmoid)

        # Temperature scaling on Perch logits before sigmoid (ref: 0-908, T=1.15)
        perch_probs = sigmoid(_logits_to_sigmoid / TEMP_SCALE)

        # ── Finetuned heads ───────────────────────────────────────────────────
        tls_h = get_perch_tflite()
        if hasattr(tls_h, 'h1'):
            head_probs = np.zeros((N_WINDOWS, NUM_CLASSES), dtype=np.float32)
            for _i in range(N_WINDOWS):
                _gathered = np.append(logits_14795[_i], 0.0).astype(np.float32)[BC_INDICES][None]
                tls_h.h1.set_tensor(tls_h.h1_inp, _gathered); tls_h.h1.invoke()
                _p1 = tls_h.h1.get_tensor(tls_h.h1_out)[0]
                tls_h.h2.set_tensor(tls_h.h2_inp, _gathered); tls_h.h2.invoke()
                _p2 = tls_h.h2.get_tensor(tls_h.h2_out)[0]
                _hsum = W_HEAD_PSEUDO * _p1 + W_HEAD_SOUNDSCAPE * _p2
                _hw   = W_HEAD_PSEUDO + W_HEAD_SOUNDSCAPE
                if hasattr(tls_h, 'h3'):
                    tls_h.h3.set_tensor(tls_h.h3_inp, emb_1536[_i:_i+1]); tls_h.h3.invoke()
                    _p3 = tls_h.h3.get_tensor(tls_h.h3_out)[0]
                    _hsum += W_HEAD_EMBEDDING * _p3; _hw += W_HEAD_EMBEDDING
                head_probs[_i] = _hsum / _hw
            perch_probs = (1.0 - HEAD_BLEND_ALPHA) * perch_probs + HEAD_BLEND_ALPHA * head_probs

    except Exception as e:
        print(f'  ERROR Perch {ogg_path.name}: {e}')
        perch_probs = np.zeros((N_WINDOWS, NUM_CLASSES), dtype=np.float32)

    if USE_SED:
        try:
            sed_probs = sed_predict_clips(clips)
        except Exception as e:
            print(f'  ERROR SED {ogg_path.name}: {e}')
            sed_probs = np.zeros((N_WINDOWS, NUM_CLASSES), dtype=np.float32)
    else:
        sed_probs = None

    if USE_SED and USE_VLOM_BLEND:
        preds = vlom_blend(perch_probs, sed_probs, w_a=PERCH_W, w_b=SED_W)
    elif USE_SED:
        preds = (PERCH_W * perch_probs + SED_W * sed_probs) / (PERCH_W + SED_W)
    else:
        preds = perch_probs

    preds = apply_texture_smooth_per_file(preds)
    preds = apply_trick2(preds)
    preds = apply_file_max(preds)

    # Co-occurrence boost: P += α*(P_max + P_mean - P_max_mean) (BirdCLEF 2024 3rd, +0.02 LB)
    if COOCCUR_BOOST_ENABLED:
        p_max  = preds.max(axis=0, keepdims=True)          # (1, 234)
        p_mean = preds.mean(axis=0, keepdims=True)          # (1, 234)
        p_max_mean = (p_max + p_mean) / 2.0
        preds  = preds + COOCCUR_BOOST_ALPHA * (p_max + p_mean - p_max_mean)
        preds  = np.clip(preds, 0.0, 1.0)

    return row_ids, preds.astype(np.float32)


NUM_THREADS = 2
print(f'Inferring {len(ogg_files)} soundscapes  threads={NUM_THREADS}')

all_row_ids, all_preds = [], []
with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
    for k, (rids, pred) in enumerate(ex.map(predict_file, ogg_files)):
        all_row_ids.extend(rids)
        all_preds.append(pred)
        if k % 50 == 0:
            elapsed = (time.time() - START) / 60
            eta     = (elapsed / (k + 1)) * (len(ogg_files) - k - 1)
            print(f'  [{k+1}/{len(ogg_files)}] {elapsed:.1f}min  eta={eta:.1f}min')

if len(all_preds) == 0:
    predictions = all_preds
else:
    predictions = np.concatenate(all_preds, axis=0)
    print(f'Done: {len(all_row_ids)} rows  {(time.time()-START)/60:.1f}min')

In [ ]:
# ── Save submission ───────────────────────────────────────────────────────────
submission = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission.to_csv('submission.csv', index=False)

print(f'submission.csv: {submission.shape[0]} rows x {len(primary_labels)} species')
print(f'Total elapsed: {(time.time()-START)/60:.1f} min')
print(f'VLOM blend: Perch={PERCH_W}  SED={SED_W}')
print(f'TRICK1={TRICK1_FILE_MAX}  TRICK2={TRICK2_SMOOTH}  TRICK3={TRICK3_NO_PEAK_NORM}')
print(f'Probe: LGBM  n_models={len(probe_models)}  features=74')
submission.head(3)
